In [1]:
import torch
import torch.nn.functional as F
import json
import sys
import numpy as np
import textwrap

In [2]:
# ── Environment setup ─────────────────────────────────────────────────────
# Run this cell first every Colab session (or once on a local machine).
# It mounts Drive (Colab only) and adds the repo root to sys.path so all
# project imports work correctly.
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Add the repo root to the module search path
sys.path.insert(0, os.path.abspath(".."))   # works locally from notebooks/
                                             # adjust to REPO_ROOT on Colab

import config as cfg
cfg.make_dirs()
cfg.print_config()

Mounted at /content/drive


In [3]:
from model import GPT, GPTWithKVCache, GPTConfig
config   = GPTConfig()
device   = "cuda" if __import__("torch").cuda.is_available() else "cpu"
model    = GPT(config).to(device)
model_kv = GPTWithKVCache(config).to(device)

cpu


In [6]:

if device == "cpu":
  ckpt = torch.load(cfg.PRETRAIN_FINAL_CKPT, map_location="cpu")
else:
  ckpt = torch.load(cfg.PRETRAIN_FINAL_CKPT)

model.load_state_dict(ckpt)
model_kv.load_state_dict(ckpt)

model.eval()
model_kv.eval()
print("Model loaded successfully")

Model loaded successfully


In [7]:
if device == "cpu":
  torch.set_num_threads(2)

In [8]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer(
    cfg.TOKENIZER_VOCAB,
    cfg.TOKENIZER_MERGES
)

In [9]:
from generation.sampler import generate, generate_kv, encode_prompt

In [10]:
%%time

prompt = "Once upon a time"

input_ids = tokenizer.encode(prompt).ids
x = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

output = generate(
    model,
    x,
    max_new_tokens=120,
    temperature=0,
    top_k=None,
    top_p=None,
    repetition_penalty=1
)
print("\n Output without KV cache")
tokens = output[0].tolist()
text = tokenizer.decode(tokens)
print(textwrap.fill(text, 80))



 Output without KV cache
Once upon a time, there was a little girl named Lily. She loved to play with her
toys and her favorite toy was a teddy bear. One day, Lily's mom asked her to
help clean up her toys. Lily didn't want to stop playing, but she knew she had
to listen to her mom.  Lily started to clean up her toys and put them away. She
picked up her teddy bear and put it in a box. She was very proud of herself for
being a good helper. She went to bed that night and had a dream about her teddy
bear.  In the dream, Lily's teddy
CPU times: user 23.7 s, sys: 3.8 ms, total: 23.7 s
Wall time: 13.8 s


In [12]:
%%time

output = generate_kv(
    model_kv,
    x,
    max_new_tokens=500,
    temperature=0.8,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.2
)

print("\n Output with KV cache")
tokens = output[0].tolist()
text = tokenizer.decode(tokens)
print(textwrap.fill(text, 80))


 Output with KV cache
Once upon a time, there was a little boy named Timmy. Timmy loved to play with
his toy cars and trucks all day long. One day, he found a red car in the park!
It was so cool and shiny that it made him smile.  As Timmy played with his blue
truck, he saw an old man sitting on a bench. The old man had a big sack of juice
in his hand. Timmy wanted to pour some juice into the juice. He asked the old
man if he could try to pour the juice into the bucket. The old man said yes, but
only for one little while.  Later that day, Timmy's mom came home from work. She
was surprised to see that he did not have any more juice left to pour. But then
she remembered something her grandma had taught him about wine. She went inside
and took out a bottle of wine. She poured some wine into a glass cup and brought
it back outside.   When she got outside, she poured it up on the grass. She
watered it and watched as the shade melted away. Then, she poured some water
into the bowl and stirre

In [14]:
%%time

prompt = "Write a story about a dragon"

input_ids = tokenizer.encode(prompt).ids
x = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

output = generate_kv(
    model_kv,
    x,
    max_new_tokens=200,
    temperature=0.8,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.2
)

print("\n Output with KV cache")
tokens = output[0].tolist()
text = tokenizer.decode(tokens)
print(textwrap.fill(text, 80))


 Output with KV cache
Write a story about a dragon. He was brave and strong.  The next day, the mouse
went back to sleep with his friends. They had a big feast of cake on the couch.
But when he woke up, he remembered what his mom said. So, he decided to go out
and look for another dragon.  When he got there, all his friends were so happy
to see him. They played games together and ate yummy food. They all became good
friends and lived happily ever after.One day, a little girl named Lily found a
mysterious box in her room. She wanted to open it, but she could not find
anything fun inside. Lily asked her mom, "Mom, can I open this secret?" Her mom
said, "Yes, Lily, you can open the lock now."  Lily tried very hard to open the
box, but it wouldn't budge. She felt sad that she couldn't open the box without
asking questions about something important. Then, her mom came into the room and
saw the
CPU times: user 7.44 s, sys: 21.4 ms, total: 7.46 s
Wall time: 4.03 s
